# DeepTaxa Tutorial
**DeepTaxa** classifies bacterial 16S rRNA gene sequences into 7 taxonomic ranks (Domain → Species) using a hybrid CNN-BERT deep learning model (92.96% species-level accuracy).

This notebook covers:
1. Installation
2. HuggingFace login
3. Downloading the pre-trained model
4. Downloading sample test data (Greengenes2)
5. Running prediction
6. Inspecting results

## Step 1 — Install DeepTaxa

In [1]:
!git clone https://github.com/systems-genomics-lab/deeptaxa.git
%cd deeptaxa
!pip install -q .
!deeptaxa --version

fatal: destination path 'deeptaxa' already exists and is not an empty directory.
/content/deeptaxa
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
DeepTaxa 0.1.0.dev1


## Step 2 — Log in to HuggingFace
You need a HuggingFace account and a token with **read** access.
Get your token at: https://huggingface.co/settings/tokens

In [3]:
from huggingface_hub import login
login()  # Enter your HuggingFace token when prompted

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Step 3 — Download Pre-trained Model
Two models are available:
- `deeptaxa-full-length-v1.pt` — full-length 16S sequences (92.96% species accuracy)
- `deeptaxa-v3v4-v1.pt` — V3/V4 hypervariable region only (87.55% species accuracy)

In [4]:
import os
os.makedirs('/content/deeptaxa-data/models', exist_ok=True)

from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='deeptaxa-full-length-v1.pt',
    local_dir='/content/deeptaxa-data/models'
)
print('Model downloaded to:', model_path)

Model downloaded to: /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt


## Step 4 — Inspect the Checkpoint

In [5]:
!deeptaxa describe \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt

2026-05-01 20:51:02,782 - INFO - 
          DeepTaxa Model Description (v0.1.0.dev1)
--------------------------------------------------
          Checkpoint: /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt
          Timestamp: 2026-05-01T20:51:02.782504

Model Details:
--------------------------------------------------
                   run-uuid: 2026_04_25T10_33_21_0c4d1cf6_44e0_4a66_a937_95560ff4c958
                 model-type: hybridcnnbert
                  tokenizer: zhihan1996/DNABERT-2-117M
                      epoch: 10
           total-parameters: 76,365,205
                 max-length: 512
                  embed-dim: 896
                num-filters: 256
               kernel-sizes: [3, 5, 7]
            num-conv-layers: 1
                hidden-size: 896
          num-hidden-layers: 4
        num-attention-heads: 7
          intermediate-size: 3,584
        hidden-dropout-prob: 0.2

Training Hyperparameters:
--------------------------------------------------
    

## Step 5 — Download Sample Test Data (Greengenes2 2024.09)

In [6]:
os.makedirs('/content/deeptaxa-data/greengenes', exist_ok=True)

from huggingface_hub import hf_hub_download
for fname in ['gg_2024_09_testing.fna.gz', 'gg_2024_09_testing.tsv.gz']:
    path = hf_hub_download(
        repo_id='systems-genomics-lab/greengenes',
        filename=fname,
        repo_type='dataset',
        local_dir='/content/deeptaxa-data/greengenes'
    )
    print('Downloaded:', path)

Downloaded: /content/deeptaxa-data/greengenes/gg_2024_09_testing.fna.gz
Downloaded: /content/deeptaxa-data/greengenes/gg_2024_09_testing.tsv.gz


## Step 6 — Preview Input Data

In [7]:
import gzip

print('=== First 3 FASTA sequences ===')
count = 0
with gzip.open('/content/deeptaxa-data/greengenes/gg_2024_09_testing.fna.gz', 'rt') as f:
    for line in f:
        print(line, end='')
        if line.startswith('>') and count > 0:
            break
        if line.startswith('>'):
            count += 1

print('\n=== First 3 taxonomy labels ===')
import pandas as pd
tax = pd.read_csv('/content/deeptaxa-data/greengenes/gg_2024_09_testing.tsv.gz', sep='\t', nrows=3)
print(tax)

=== First 3 FASTA sequences ===
>MJ031-2-barcode64-umi2692bins-ubs-145
ACAAACGCCAGCGGCGTGCCTAACACATGCAAGTCGAACGAGAAATCCTGAGCAATCGGG
AGAGTAAAGTGGCGCACGGGTGAGTAACGCGTGGGTAATCTACCCTGGAGTTGGGAATAA
CATCTCGAAAGGGGTGCTAATACCTAATAAAATCTTTGGGGCTCCGGTCTTGGAGATTAA
AGAGGGCTGAGGCTTGCAAGCCAGTGCTCAAGGATGAGCCCGCGTACCATTAGCTAGTTG
GTGGGGTAATGGCCCACCAAGGCAATGATGGTTAGCTGGTCTGAGAGGATGGCCAGCCAC
ACTGGAACTGAGATACGGTCCAGACTCCTACGGGAGGCAGCAGTGAGGAATCTTGCGCAA
TGGGGGAAACCCTGACGCAGCAACGCCGCGTGAGTGATGAAGGCCTTCGGGTCGTAAAGC
TCTGTCAGATGGGAAGAAAGTTCCGGCGATGAATAAGCGCTGGAATTGACGGTACCGTTA
AAGGAAGCACCGGCTAACTCCGTGCCAGCAGCCGCGGTAATACGGAGGGTGCAAGCGTTG
TTCGGAATTATTGGGCGTAAAGAGCGTGTAGGCGGCCTAGTAAGTCAGATGTGAAAGCCC
TTGGCTTAACCAAGGAAGGGCATTTGAAACTGCTTGGCTTGAGTACGGGAGGGGGAAGCG
GAATTCCCGGTGTAGAGGTGAAATTCGTAGATATCGGGAGGAACACCGGTGGCGAAGGCG
GCTTCCTGGACCGATACTGACGCTGAGACGCGAAAGCGTGGGGAGCAAACAGGATTAGAT
ACCCTGGTAGTCCACGCCATAAACGTTGGGCACTAGGTGTTGCGGGTATTGACCCCTGCA
GTGCCGAAGCTAACGCATTAAGTGCCCCGCCTGGGGAGTACGGTCGCAAGGCTAAAACTC
AAAGGAATTGACGG

## Step 7 — Run Prediction with Evaluation
Since we have the true taxonomy labels, this will compute accuracy at all 7 ranks.

In [ ]:
os.makedirs('/content/deeptaxa-outputs/predictions', exist_ok=True)

!deeptaxa predict \
    --fasta-file /content/deeptaxa-data/greengenes/gg_2024_09_testing.fna.gz \
    --taxonomy-file /content/deeptaxa-data/greengenes/gg_2024_09_testing.tsv.gz \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt \
    --output-dir /content/deeptaxa-outputs/predictions

2026-05-01 20:51:14,900 - INFO - 
          DeepTaxa Prediction Session (v0.1.0.dev1)
--------------------------------------------------
          Started: 2026-05-01T20:51:14.900182
    
2026-05-01 20:51:15,386 - INFO - HTTP Request: HEAD https://huggingface.co/zhihan1996/DNABERT-2-117M/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-01 20:51:15,488 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/config.json "HTTP/1.1 200 OK"
2026-05-01 20:51:15,580 - INFO - HTTP Request: HEAD https://huggingface.co/zhihan1996/DNABERT-2-117M/resolve/main/configuration_bert.py "HTTP/1.1 307 Temporary Redirect"
2026-05-01 20:51:15,688 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/zhihan1996/DNABERT-2-117M/7bce263b15377fc15361f52cfab88f8b586abda0/configuration_bert.py "HTTP/1.1 200 OK"
2026-05-01 20:51:15,777 - INFO - HTTP Request: HEAD https://huggingface.co/zhi

## Step 8 — Inspect Results

In [ ]:
import os, json

output_dir = '/content/deeptaxa-outputs/predictions'
print('Output files:')
for f in os.listdir(output_dir):
    print(' ', f)

# Load and display prediction results
pred_files = [f for f in os.listdir(output_dir) if f.endswith('.tsv') or f.endswith('.csv')]
if pred_files:
    df = pd.read_csv(os.path.join(output_dir, pred_files[0]), sep='\t')
    print('\nPrediction sample (first 5 rows):')
    print(df.head())

# Load metrics if available
metric_files = [f for f in os.listdir(output_dir) if f.endswith('.json')]
if metric_files:
    with open(os.path.join(output_dir, metric_files[0])) as f:
        metrics = json.load(f)
    print('\nAccuracy by taxonomic rank:')
    for rank, val in metrics.items():
        if 'accuracy' in rank.lower() or 'acc' in rank.lower():
            print(f'  {rank}: {val:.4f}')

In [ ]:
import os, glob
os.chdir(os.path.expanduser('~/deeptaxa-workspace'))
pred_files = sorted(glob.glob('outputs/predictions/*_predictions.tsv'))
for f in pred_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f'{f} ({size_mb:.1f} MB)')

## (Optional) Step 9 — Predict on Your Own Sequences
Upload a FASTA file with your own 16S rRNA sequences and run prediction.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload your .fna or .fna.gz file

fasta_name = list(uploaded.keys())[0]
os.makedirs('/content/deeptaxa-outputs/my_predictions', exist_ok=True)

!deeptaxa predict \
    --fasta-file /content/{fasta_name} \
    --checkpoint /content/deeptaxa-data/models/deeptaxa-full-length-v1.pt \
    --output-dir /content/deeptaxa-outputs/my_predictions

print('\nResults:')
for f in os.listdir('/content/deeptaxa-outputs/my_predictions'):
    print(' ', f)